# Tutorial: DNA Embeddings & Variant-Effect Analysis with embpy

This tutorial walks through three workflows:

1. **Raw sequence embedding** — embed any DNA string and compare models side-by-side
2. **Gene-level embedding** — resolve human genes by symbol / Ensembl ID and embed them
3. **Variant-effect embedding** — quantify the effect of SNPs using reference vs alternate context embeddings

All examples use real pre-trained models downloaded from HuggingFace the first time they are run.

---
**Installation**

```bash
# CPU
pip install embpy

# GPU (CUDA 12.4)
pip install "embpy[torch-cu124]" --extra-index-url https://download.pytorch.org/whl/cu124

# Caduceus (optional — requires CUDA and mamba-ssm)
pip install mamba-ssm --no-build-isolation
```

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## Part 1 — Raw Sequence Embedding

Every DNA wrapper exposes the same two-step interface:
```python
wrapper = SomeWrapper(model_path_or_name="...")
wrapper.load(device)                          # downloads weights once, caches
embedding = wrapper.embed(sequence, pooling_strategy="mean")  # → np.ndarray
```

We will embed five biologically distinct sequences and visualise them.

In [ ]:
SEQUENCES = {
    "TP53 exon 7":   "CCTGCATGGGCGGCATGAACCGGAGGCCCATCCTCACCATCATCACACTGGAAGACTCCAGTGGTAATCTACTGGG"
                     "ACGGAACAGCTTTGAGGTGCGTGTTTGTGCCTGTCCTGGGAGAGACCGGCGCACAGAGGAAGAGAATCTCCGCAA",
    "BRCA1 exon 11": "GAAGATATTTCTGGGAGCCAGCTGCTCTGAGACTCAGAGAGAGGGAGAAGCAGAAGCAGCAGCAGCAGCAGCAGCA"
                     "GCAGCAGCAGCAGCAGCAGCAGCAGCAGCAG",
    "KRAS exon 2":   "GTATTTATTCAGTCAAGCCTTTGCAGCAAGACAAGAATGACTGAATATAAACTTGTGGTAGTTGGAGCTGGTGGCG"
                     "TAGGCAAGAGTGCCTTGACGATACAG",
    "poly-A repeat": "A" * 150,
    "CpG island":    "CGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCG"
                     "CGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCGCG",
}

print("Sequences loaded:")
for name, seq in SEQUENCES.items():
    print(f"  {name:<20} {len(seq):>4} bp")

### 1.1  Load models

In [ ]:
from embpy.models.dna_models import (
    GENALMWrapper,
    NucleotideTransformerWrapper,
    HyenaDNAWrapper,
)

models = {
    "GENA-LM": GENALMWrapper("AIRI-Institute/gena-lm-bert-base-t2t"),
    "NT v2 100M": NucleotideTransformerWrapper(
        "InstaDeepAI/nucleotide-transformer-v2-100m-multi-species"
    ),
    "HyenaDNA": HyenaDNAWrapper("LongSafari/hyenadna-small-32k-seqlen-hf"),
}

for name, wrapper in models.items():
    print(f"Loading {name} ...")
    wrapper.load(device)
    print(f"  ✓ {name} ready")

### 1.2  Embed sequences and inspect shapes

In [ ]:
all_embeddings = {}   # model_name → {seq_name: np.ndarray}

for model_name, wrapper in models.items():
    embs = {}
    for seq_name, seq in SEQUENCES.items():
        embs[seq_name] = wrapper.embed(seq, pooling_strategy="mean")
    all_embeddings[model_name] = embs
    dim = list(embs.values())[0].shape[0]
    print(f"{model_name:<15} embedding dim = {dim}")

### 1.3  Pooling strategy comparison

All models support `mean`, `max`, and `cls` pooling.  HyenaDNA additionally supports `last`.

In [ ]:
seq = SEQUENCES["TP53 exon 7"]
wrapper = models["NT v2 100M"]
strategies = ["mean", "max", "cls"]

print("NT v2 pooling strategies on TP53 exon 7:")
pooled = {}
for s in strategies:
    e = wrapper.embed(seq, pooling_strategy=s)
    pooled[s] = e
    print(f"  {s:<6}  shape={e.shape}  L2={np.linalg.norm(e):.3f}")

# Cosine similarity between strategies
print("\nCosine similarities between pooling strategies:")
for i, s1 in enumerate(strategies):
    for s2 in strategies[i+1:]:
        cos = float(cosine_similarity(pooled[s1][None], pooled[s2][None])[0, 0])
        print(f"  {s1} vs {s2}: {cos:.4f}")

### 1.4  PCA visualisation — do models agree on sequence similarity?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
seq_names = list(SEQUENCES.keys())
colors = ["#e06c75", "#61afef", "#98c379", "#e5c07b", "#c678dd"]

for ax, (model_name, embs) in zip(axes, all_embeddings.items()):
    X = np.stack([embs[n] for n in seq_names])
    pca = PCA(n_components=2)
    coords = pca.fit_transform(X)

    for i, (name, color) in enumerate(zip(seq_names, colors)):
        ax.scatter(*coords[i], s=100, color=color, edgecolors="white", linewidths=0.8, zorder=3)
        ax.annotate(name, coords[i], textcoords="offset points", xytext=(5, 3), fontsize=7.5)

    ax.set_title(model_name, fontweight="bold")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}%)")
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("PCA of sequence embeddings (mean pooling)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 1.5  Intermediate layer embeddings

Pass `target_layer=N` to extract representations from a specific transformer layer
instead of the final layer.

In [ ]:
wrapper = models["GENA-LM"]
seq = SEQUENCES["TP53 exon 7"]

layer_embs = {}
for layer in [0, 3, 6, 9, 11]:   # GENA-LM bert-base has 12 layers (0-11)
    e = wrapper.embed(seq, pooling_strategy="mean", target_layer=layer)
    layer_embs[layer] = e
    print(f"  Layer {layer:2d}  L2={np.linalg.norm(e):.3f}")

# Cosine similarity of each layer vs layer 11 (final)
print("\nCosine similarity vs final layer:")
final = layer_embs[11]
for layer, e in sorted(layer_embs.items()):
    cos = float(cosine_similarity(e[None], final[None])[0, 0])
    print(f"  Layer {layer:2d}: {cos:.4f}")

---
## Part 2 — Gene-Level Embeddings

`BioEmbedder` resolves gene identifiers to DNA sequences automatically using the
Ensembl REST API, then passes the sequence to the chosen model.

In [ ]:
from embpy.embedder import BioEmbedder

embedder = BioEmbedder(device="auto", resolver_backend="api")

# List available DNA model keys
dna_models = [
    m for m in embedder.list_available_models()
    if any(k in m for k in ["gena", "nt_v", "nt_500", "nt_2b", "hyena", "caduceus"])
]
print("Available DNA models:")
for m in dna_models:
    print(f"  {m}")

In [ ]:
# Embed canonical cancer genes by symbol, using NT v2
cancer_genes = ["TP53", "BRCA1", "KRAS", "MYC", "EGFR", "PTEN", "RB1", "APC"]

gene_embeddings = {}
for gene in cancer_genes:
    try:
        emb = embedder.embed_gene(gene, model="nt_v2_100m", id_type="symbol")
        gene_embeddings[gene] = emb
        print(f"  {gene:<8} shape={emb.shape}  L2={np.linalg.norm(emb):.3f}")
    except Exception as e:
        print(f"  {gene:<8} FAILED — {e}")

In [ ]:
# PCA + cosine heatmap
if len(gene_embeddings) >= 4:
    names = list(gene_embeddings.keys())
    X = np.stack(list(gene_embeddings.values()))

    fig, (ax_pca, ax_heat) = plt.subplots(1, 2, figsize=(12, 5))

    # PCA
    pca = PCA(n_components=2)
    coords = pca.fit_transform(X)
    scatter_colors = plt.cm.tab10(np.linspace(0, 1, len(names)))  # type: ignore
    for i, (name, c) in enumerate(zip(names, scatter_colors)):
        ax_pca.scatter(*coords[i], s=120, color=c, edgecolors="white", linewidths=0.8, zorder=3)
        ax_pca.annotate(name, coords[i], textcoords="offset points", xytext=(6, 4), fontsize=10)
    ax_pca.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax_pca.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    ax_pca.set_title("Gene embeddings — PCA (NT v2 100M)", fontweight="bold")
    ax_pca.spines[["top", "right"]].set_visible(False)

    # Cosine heatmap
    cos_mat = cosine_similarity(X)
    im = ax_heat.imshow(cos_mat, vmin=0, vmax=1, cmap="YlOrRd")
    ax_heat.set_xticks(range(len(names))); ax_heat.set_xticklabels(names, rotation=45, ha="right")
    ax_heat.set_yticks(range(len(names))); ax_heat.set_yticklabels(names)
    for i in range(len(names)):
        for j in range(len(names)):
            ax_heat.text(j, i, f"{cos_mat[i,j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax_heat, label="Cosine similarity")
    ax_heat.set_title("Pairwise cosine similarity", fontweight="bold")

    plt.tight_layout()
    plt.show()

---
## Part 3 — SNP / Variant-Effect Embeddings

The `SNPEmbedder` class computes **variant-effect embeddings** by comparing how a
model represents the genomic context before and after a nucleotide substitution.

For each SNP it produces:

| Quantity | Description |
|---|---|
| `ref_embedding` | Embedding of the reference context window |
| `alt_embedding` | Embedding with the variant allele substituted in |
| `delta_embedding` | `alt_emb − ref_emb` |
| `delta_norm` | ‖alt − ref‖₂  (scalar effect size) |
| `cosine_similarity` | cos(ref, alt) |

### 3.1  Download the chromosome 17 reference sequence

In [ ]:
from pathlib import Path
from embpy.tl.snp_utils import download_hg38_per_chrom, SequenceProvider

FASTA_DIR = Path("./hg38_chroms")

# ~100 MB download; skipped automatically if already present
download_hg38_per_chrom(output_dir=FASTA_DIR, chromosomes=["chr17"])

provider = SequenceProvider(fasta_dir=str(FASTA_DIR))
chr17 = provider.get_chromosome("chr17")
print(f"chr17 loaded: {len(chr17):,} bp")

### 3.2  Define TP53 test variants

We use three characterised TP53 variants covering two extremes:
cancer hotspot mutations (large functional impact) and a common germline
polymorphism (modest functional impact).

| rsID | Protein change | Classification |
|---|---|---|
| rs28934578 | p.R175H | Frequent somatic cancer hotspot |
| rs28934574 | p.R248W | Frequent somatic cancer hotspot |
| rs1042522 | p.R72P | Common germline polymorphism (MAF ~40%) |

In [ ]:
TP53_VARIANTS = [
    {"variant_id": "rs28934578", "chrom": "chr17", "pos": 7_674_220,
     "ref": "C", "alt": "T", "label": "R175H (hotspot)"},
    {"variant_id": "rs28934574", "chrom": "chr17", "pos": 7_673_802,
     "ref": "G", "alt": "A", "label": "R248W (hotspot)"},
    {"variant_id": "rs1042522",  "chrom": "chr17", "pos": 7_676_154,
     "ref": "C", "alt": "G", "label": "R72P (polymorphism)"},
]

CONTEXT_WINDOW = 256

### 3.3  Embed variants with all three models

In [ ]:
from embpy.tl.snp_utils import SNPEmbedder

snp_results = {}   # model_name → list[SNPEmbeddingResult]

for model_name, wrapper in models.items():
    snp_emb = SNPEmbedder(wrapper, pooling_strategy="mean")
    results = []
    for v in TP53_VARIANTS:
        r = snp_emb.embed_snp_from_vcf_row(
            chrom=v["chrom"], pos=v["pos"], ref=v["ref"], alt=v["alt"],
            chromosome_sequence=chr17, context_window=CONTEXT_WINDOW,
            variant_id=v["variant_id"],
        )
        results.append(r)
    snp_results[model_name] = results

# Summary table
print(f"{'Model':<15} {'Variant':<14} {'Label':<24} {'Delta L2':>10} {'Cosine':>10}")
print("-" * 75)
for model_name, results in snp_results.items():
    for r, v in zip(results, TP53_VARIANTS):
        print(
            f"{model_name:<15} {v['variant_id']:<14} {v['label']:<24}"
            f"{r.delta_norms[0]:>10.4f} {r.cosine_similarities[0]:>10.6f}"
        )

### 3.4  Visualise variant effect size

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
model_names = list(snp_results.keys())
variant_labels = [v["variant_id"] for v in TP53_VARIANTS]
variant_clinical = [v["label"] for v in TP53_VARIANTS]
x = np.arange(len(variant_labels))
w = 0.25
bar_colors = ["#4C72B0", "#DD8452", "#55A868"]

# Delta L2 norms
ax = axes[0]
for i, (model_name, color) in enumerate(zip(model_names, bar_colors)):
    deltas = [snp_results[model_name][j].delta_norms[0] for j in range(len(TP53_VARIANTS))]
    ax.bar(x + i * w, deltas, width=w, label=model_name, color=color, edgecolor="white")
ax.set_xticks(x + w); ax.set_xticklabels(variant_labels)
ax.set_ylabel("Delta L2 norm  ‖alt − ref‖")
ax.set_title("Variant effect size", fontweight="bold")
ax.legend(frameon=False)
for i, label in enumerate(variant_clinical):
    ax.text(x[i] + w, ax.get_ylim()[1] * 0.98, label,
            ha="center", va="top", fontsize=7.5, color="gray", style="italic")
ax.spines[["top", "right"]].set_visible(False)

# 1 − cosine similarity
ax = axes[1]
for i, (model_name, color) in enumerate(zip(model_names, bar_colors)):
    dists = [1.0 - snp_results[model_name][j].cosine_similarities[0] for j in range(len(TP53_VARIANTS))]
    ax.bar(x + i * w, dists, width=w, label=model_name, color=color, edgecolor="white")
ax.set_xticks(x + w); ax.set_xticklabels(variant_labels)
ax.set_ylabel("1 − cosine(ref, alt)")
ax.set_title("Cosine distance", fontweight="bold")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("TP53 variant effects — three DNA foundation models", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 3.5  Embedding-space view of ref vs alt

Project all reference and alternate embeddings into 2D with PCA.
Each variant is shown as a pair: circle (ref) and triangle (alt), connected by a dashed line.
Longer lines = larger variant effect.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=False)
variant_colors = ["#e06c75", "#61afef", "#98c379"]

for ax, model_name in zip(axes, model_names):
    results = snp_results[model_name]
    refs = np.stack([r.ref_embedding for r in results])
    alts = np.stack([r.alt_embeddings[0] for r in results])
    all_embs = np.concatenate([refs, alts], axis=0)

    pca2 = PCA(n_components=2)
    coords = pca2.fit_transform(all_embs)
    n = len(results)

    for i, (v, c) in enumerate(zip(TP53_VARIANTS, variant_colors)):
        ax.scatter(*coords[i],     s=110, color=c, marker="o", zorder=3,
                   label=f"{v['variant_id']} ref")
        ax.scatter(*coords[n + i], s=110, color=c, marker="^", zorder=3,
                   label=f"{v['variant_id']} alt")
        ax.annotate(v['variant_id'], coords[i],
                    textcoords="offset points", xytext=(5, 3), fontsize=7.5)
        ax.plot([coords[i, 0], coords[n+i, 0]], [coords[i, 1], coords[n+i, 1]],
                color=c, alpha=0.5, lw=1.5, ls="--")

    ax.set_title(model_name, fontweight="bold")
    ax.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]*100:.0f}%)")
    ax.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]*100:.0f}%)")
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Ref (○) vs Alt (△) — TP53 variants in embedding space",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 3.6  Context window sensitivity

How does the size of the context window around the SNP affect the variant-effect signal?

In [ ]:
hotspot = TP53_VARIANTS[0]   # R175H
context_sizes = [32, 64, 128, 256, 512]

fig, axes = plt.subplots(1, len(models), figsize=(14, 3.5), sharey=False)

for ax, (model_name, wrapper) in zip(axes, models.items()):
    snp_emb = SNPEmbedder(wrapper, pooling_strategy="mean")
    deltas = []
    for ctx in context_sizes:
        r = snp_emb.embed_snp_from_vcf_row(
            chrom=hotspot["chrom"], pos=hotspot["pos"],
            ref=hotspot["ref"], alt=hotspot["alt"],
            chromosome_sequence=chr17, context_window=ctx,
            variant_id=hotspot["variant_id"],
        )
        deltas.append(r.delta_norms[0])
    ax.plot(context_sizes, deltas, marker="o", color="#4C72B0", lw=2)
    ax.set_xlabel("Context window (bp)")
    ax.set_ylabel("Delta L2 norm")
    ax.set_title(model_name, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"Context window sensitivity — {hotspot['variant_id']} ({hotspot['label']})",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### 3.7  Batch VCF processing

`embed_vcf()` reads a standard VCF file and embeds all variants at once.
Here we demonstrate with a small in-memory VCF.

In [ ]:
import tempfile, os
from embpy.tl.snp_utils import embed_vcf

VCF_CONTENT = """##fileformat=VCFv4.1
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO
chr17\t7674220\trs28934578\tC\tT\t.\t.\t.
chr17\t7673802\trs28934574\tG\tA\t.\t.\t.
chr17\t7676154\trs1042522\tC\tG\t.\t.\t.
"""

with tempfile.NamedTemporaryFile(suffix=".vcf", mode="w", delete=False) as f:
    f.write(VCF_CONTENT)
    vcf_path = f.name

vcf_results = embed_vcf(
    vcf_path=vcf_path,
    model_wrapper=models["NT v2 100M"],
    chromosome_sequences={"chr17": chr17},
    context_window=256,
    pooling_strategy="mean",
)
os.unlink(vcf_path)

print(f"Embedded {len(vcf_results)} variants from VCF")
for r in vcf_results:
    print(f"  {r.snp.variant_id:<14}  delta_L2={r.delta_norms[0]:.4f}  cosine={r.cosine_similarities[0]:.6f}")

---
## Part 4 — Saving and Loading Results

In [ ]:
# Save SNP results to .npz for downstream analysis
results_to_save = snp_results["NT v2 100M"]

np.savez(
    "tp53_snp_embeddings.npz",
    variant_ids      = np.array([r.snp.variant_id for r in results_to_save], dtype=str),
    ref_embeddings   = np.stack([r.ref_embedding   for r in results_to_save]),
    alt_embeddings   = np.stack([r.alt_embeddings[0] for r in results_to_save]),
    delta_embeddings = np.stack([r.delta_embeddings[0] for r in results_to_save]),
    delta_norms      = np.array([r.delta_norms[0]  for r in results_to_save]),
    cosine_sims      = np.array([r.cosine_similarities[0] for r in results_to_save]),
)

# Reload and verify
saved = np.load("tp53_snp_embeddings.npz", allow_pickle=True)
print("Saved to tp53_snp_embeddings.npz")
print(f"  variant_ids:       {saved['variant_ids']}")
print(f"  ref_embeddings:    {saved['ref_embeddings'].shape}")
print(f"  delta_embeddings:  {saved['delta_embeddings'].shape}")
print(f"  delta_norms:       {saved['delta_norms'].round(4)}")

---
## Quick Reference

### DNA model registry keys

| Key | Model | Params | Max context |
|---|---|---:|---:|
| `gena_lm_bert_base` | GENA-LM BERT-base | 110M | 4.5 kb |
| `gena_lm_bert_large` | GENA-LM BERT-large | 336M | 4.5 kb |
| `gena_lm_bigbird_base` | GENA-LM BigBird-base | 110M | 36 kb |
| `nt_v2_50m` | NT v2 50M | 50M | 12 kb |
| `nt_v2_100m` | NT v2 100M | 100M | 12 kb |
| `nt_v2_500m` | NT v2 500M | 500M | 12 kb |
| `hyenadna_small_32k` | HyenaDNA small | 7M | 32 kb |
| `hyenadna_large_1m` | HyenaDNA large | 6.5M | 1 Mb |
| `caduceus_ph_131k` | Caduceus-PH | 7.7M | 131 kb |
| `caduceus_ps_131k` | Caduceus-PS | 7.7M | 131 kb |

### API cheat-sheet

```python
# 1. Load a wrapper directly
from embpy.models.dna_models import NucleotideTransformerWrapper
nt = NucleotideTransformerWrapper("InstaDeepAI/nucleotide-transformer-v2-100m-multi-species")
nt.load(torch.device("cuda"))
emb = nt.embed("ACGTACGT...", pooling_strategy="mean")  # (512,)

# 2. Embed a gene by symbol via BioEmbedder
from embpy.embedder import BioEmbedder
embedder = BioEmbedder(device="auto")
emb = embedder.embed_gene("TP53", model="nt_v2_100m")

# 3. SNP variant-effect embedding
from embpy.tl.snp_utils import SNPEmbedder, SequenceProvider, download_hg38_per_chrom
download_hg38_per_chrom("./hg38_chroms", chromosomes=["chr17"])
chr17 = SequenceProvider(fasta_dir="./hg38_chroms").get_chromosome("chr17")
result = SNPEmbedder(nt).embed_snp_from_vcf_row(
    chrom="chr17", pos=7_674_220, ref="C", alt="T",
    chromosome_sequence=chr17, context_window=256,
)
print(result.delta_norms)          # [0.3821]
print(result.cosine_similarities)  # [0.9987]

# 4. Batch VCF
from embpy.tl.snp_utils import embed_vcf
results = embed_vcf(
    vcf_path="variants.vcf",
    model_wrapper=nt,
    chromosome_sequences={"chr17": chr17},
    context_window=256,
)
```